In [38]:
""" Loading the data """
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras

emotion_df = pd.read_csv('C:/Users/tobos/OneDrive/Desktop/Projects/NLP-Sentiment-Analysis-project/nlp-sentiment-analysis/Datasets/emotions.csv')
emotion_df.head()

,Unnamed: 0,text,label
0,0,i just feel really helpless and heavy hearted,4
1,1,ive enjoyed being able to slouch about relax a...,0
2,2,i gave up my internship with the dmrg and am f...,4
3,3,i dont know i feel so lost,0
4,4,i am a kindergarten teacher and i am thoroughl...,4


In [3]:
gbv_hate_speech_df = pd.read_csv('C:/Users/tobos/OneDrive/Desktop/Projects/NLP-Sentiment-Analysis-project/nlp-sentiment-analysis/Datasets/gbv_hate_speech.csv')
gbv_hate_speech_df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [4]:
violence_df = pd.read_csv('C:/Users/tobos/OneDrive/Desktop/Projects/NLP-Sentiment-Analysis-project/nlp-sentiment-analysis/Datasets/violence.csv')
violence_df.head()

,Tweet_ID,tweet,type
0,ID_0022DWKP,Had a dream i got raped last night. By a guy i...,sexual_violence
1,ID_00395QYM,he thought the word raped means sex and told m...,sexual_violence
2,ID_003EOSSF,She NOT TALKING TO ME I WAS RAPED BY 2 MEN 1 M...,sexual_violence
3,ID_004BBHOD,I was sexually abused for 3 years at age 4 to ...,sexual_violence
4,ID_004F7516,Chessy Prout can do better by telling the trut...,sexual_violence


In [5]:
""" Data Preprocessing """
# Dropping unwanted columns
emotion_df.drop(columns = ['Unnamed: 0'], inplace = True)
violence_df.drop(columns = ['Tweet_ID'], inplace = True)
gbv_hate_speech_df = gbv_hate_speech_df[['tweet', 'class']]

In [6]:
emotion_df.head()

,text,label
0,i just feel really helpless and heavy hearted,4
1,ive enjoyed being able to slouch about relax a...,0
2,i gave up my internship with the dmrg and am f...,4
3,i dont know i feel so lost,0
4,i am a kindergarten teacher and i am thoroughl...,4


In [7]:
violence_df.head()

,tweet,type
0,Had a dream i got raped last night. By a guy i...,sexual_violence
1,he thought the word raped means sex and told m...,sexual_violence
2,She NOT TALKING TO ME I WAS RAPED BY 2 MEN 1 M...,sexual_violence
3,I was sexually abused for 3 years at age 4 to ...,sexual_violence
4,Chessy Prout can do better by telling the trut...,sexual_violence


In [8]:
gbv_hate_speech_df.head()

,tweet,class
0,!!! RT @mayasolovely: As a woman you shouldn't...,2
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,1
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,1
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,1
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,1


In [9]:
# Renaming the columns
violence_df.rename(columns = {'tweet' : 'text', 'type' : 'label'}, inplace = True)
gbv_hate_speech_df.rename(columns = {'tweet' : 'text', 'class' : 'label'}, inplace = True)

violence_df.columns, gbv_hate_speech_df.columns, emotion_df.columns

(Index(['text', 'label'], dtype='object'),
 Index(['text', 'label'], dtype='object'),
 Index(['text', 'label'], dtype='object'))

In [10]:
# Checking for null values
violence_df.isna().sum(), gbv_hate_speech_df.isna().sum(), emotion_df.isna().sum()

(text     0
 label    0
 dtype: int64,
 text     0
 label    0
 dtype: int64,
 text     0
 label    0
 dtype: int64)

In [51]:
# Data sampling: extracting 12 thousand rows from each dataset.
emtn_df = pd.DataFrame()
for i in range(6):
    subset = emotion_df[emotion_df['label'] == i].sample(n = 2000, random_state = 42)
    emtn_df = pd.concat([emtn_df, subset])

emotion_df = emtn_df.copy()
emotion_df['label'].value_counts()
emotion_df.shape

(12000, 3)

In [12]:
violence_df['label'].value_counts()

label
sexual_violence                 32648
Physical_violence                5946
emotional_violence                651
economic_violence                 217
Harmful_Traditional_practice      188
Name: count, dtype: int64

In [13]:
# Because violence_df has uneven counts in the columns, we only sample from the highest(sexual_violence).
sexual_violence = violence_df[violence_df['label'] == 'sexual_violence'].sample(n = 4998, random_state = 42)

# Removing 'sexual_violence' column from violence_df
violence_df = violence_df[violence_df['label'] != 'sexual_violence']

# Re-adding sampled 'sexual_violence' column to violence_df
violence_df = pd.concat( [sexual_violence, violence_df], axis = 0)

violence_df.shape

(12000, 2)

In [14]:
gbv_hate_speech_df['label'].value_counts()

label
1    19190
2     4163
0     1430
Name: count, dtype: int64

In [15]:
# Because gbv_hate_speech_df has uneven counts in the columns, we only sample from the highest(offensive_speech) column.
offensive_speech = gbv_hate_speech_df[gbv_hate_speech_df['label'] == 1].sample(n = 6407, random_state = 42)

# Removing 'offensive_speech' from gbv_hate_speech_df
gbv_hate_speech_df = gbv_hate_speech_df[gbv_hate_speech_df['label'] != 1]

# Re-adding sampled 'offensive_speech' column to gbv_hate_speech_df
gbv_hate_speech_df = pd.concat([offensive_speech, gbv_hate_speech_df], axis = 0)

gbv_hate_speech_df.shape

(12000, 2)

In [16]:
# Resetting indexes of dataframes
emotion_df.reset_index(drop = True, inplace = True)
gbv_hate_speech_df.reset_index(drop = True, inplace = True)
violence_df.reset_index(drop = True, inplace = True)

In [17]:
emotion_df.head(3)

,text,label
0,ive learned to surround myself with women who ...,0
1,i already feel crappy because of this and you ...,0
2,i feel like i have lost mourned and moved past...,0


In [18]:
gbv_hate_speech_df.head(3)

,text,label
0,Why is it everytime I go to cracker barrel the...,1
1,"Run that nigga, you don't want that nigga, but...",1
2,I need a girl from Jamaica I can't fuck with t...,1


In [19]:
violence_df.head(3)

,text,label
0,My cousin was raped by this guy Matthew. She w...,sexual_violence
1,HAHAHAHAHAHAHHA I DIDN’T SEE IT THE FIRST TIME...,sexual_violence
2,I remember how I almost got raped like it was ...,sexual_violence


In [20]:
""" Label encoding """
label_encoder = LabelEncoder()
violence_df['label'] = label_encoder.fit_transform(violence_df['label'])

violence_df.head()

,text,label
0,My cousin was raped by this guy Matthew. She w...,4
1,HAHAHAHAHAHAHHA I DIDN’T SEE IT THE FIRST TIME...,4
2,I remember how I almost got raped like it was ...,4
3,He raped me 👈,4
4,"A woman raped by A male: psychological horror,...",4


In [22]:
""" Stopwords Removal """
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tobos\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tobos\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [25]:
# Loading the stopwords
stop_words = set(stopwords.words('english'))

len(stop_words)

198

In [27]:
# Stopwords removal function
def remove_stopwords(text):
    all_words = nltk.word_tokenize(text)
    filtered_words = [word for word in all_words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

emotion_df['text'] = emotion_df['text'].apply(remove_stopwords)
violence_df['text'] = violence_df['text'].apply(remove_stopwords)
gbv_hate_speech_df['text'] = gbv_hate_speech_df['text'].apply(remove_stopwords)

emotion_df.head(5)

,Unnamed: 0,text,label
0,0,feel really helpless heavy hearted,4
1,1,ive enjoyed able slouch relax unwind frankly n...,0
2,2,gave internship dmrg feeling distraught,4
3,3,dont know feel lost,0
4,4,kindergarten teacher thoroughly weary job take...,4


In [64]:
""" Tokenization & Padding """
tokenizer = Tokenizer()
tokenizer.fit_on_texts(pd.concat([violence_df['text'],emotion_df['text'],  gbv_hate_speech_df['text']]))

emotion_sequences = tokenizer.texts_to_sequences(emotion_df['text'])
violence_sequences = tokenizer.texts_to_sequences(violence_df['text'])
gbv_hate_speech_sequences = tokenizer.texts_to_sequences(gbv_hate_speech_df['text'])

In [32]:
emotion_df['text'].iloc[2]

'i gave up my internship with the dmrg and am feeling distraught'

In [33]:
emotion_sequences[2:3]

[[1, 794, 43, 10, 6555, 24, 5, 46034, 3, 23, 7, 1213]]

In [65]:
max_length = 50
emotion_padded = pad_sequences(emotion_sequences, maxlen = max_length, padding = 'post')
violence_padded = pad_sequences(violence_sequences, maxlen = max_length, padding = 'post')
gbv_hate_speech_padded = pad_sequences(gbv_hate_speech_sequences, maxlen = max_length, padding = 'post')

emotion_padded[2:3]

array([[    1,     2,     9,     1,    32,   406, 12864,     3,  1317,
          519,     5,  1202,    21,    38,   482,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0]], dtype=int32)

In [66]:
# Generating labels in numpy array format
emotion_labels = np.array(emotion_df['label'])
violence_labels = np.array(violence_df['label'])
gbv_hate_speech_labels = np.array(gbv_hate_speech_df['label'])

In [67]:
""" Model definition  """

# Prepare separate inputs for each dataset
emotion_input = emotion_padded
violence_input = violence_padded
gbv_hate_speech_input = gbv_hate_speech_padded

# Define multiple input layers for each task
emotion_input_layer = keras.layers.Input(shape = (max_length, ), name = 'emotion_input')
violence_input_layer = keras.layers.Input(shape = (max_length, ), name = 'violence_input')
gbv_hate_speech_input_layer = keras.layers.Input(shape = (max_length, ), name = 'gbv_hate_speech_input')

# Use a shared embedding layer
embedding_layer = keras.layers.Embedding(input_dim = len(tokenizer.word_index) + 1, output_dim = 128)

# Apply the embedding layer to each input
emotion_embedding = embedding_layer(emotion_input_layer)
violence_embedding = embedding_layer(violence_input_layer)
gbv_hate_speech_embedding = embedding_layer(gbv_hate_speech_input_layer)

# Shared LSTM layer
shared_lstm = keras.layers.LSTM(64, return_sequences = True)

emotion_lstm = shared_lstm(emotion_embedding)
violence_lstm = shared_lstm(violence_embedding)
gbv_hate_speech_lstm = shared_lstm(gbv_hate_speech_embedding)

# Share global average pooling layer and dropout layer
shared_pooling = keras.layers.GlobalAveragePooling1D()
shared_dropout = keras.layers.Dropout(0.5)

emotion_features = shared_dropout(shared_pooling(emotion_lstm))
violence_features = shared_dropout(shared_pooling(violence_lstm))
gbv_hate_speech_features = shared_dropout(shared_pooling(gbv_hate_speech_lstm))

# Output layers
emotion_output = keras.layers.Dense(6, activation = 'softmax', name = 'emotion_output')(emotion_features)
violence_output = keras.layers.Dense(5, activation = 'softmax', name = 'violence_output')(violence_features)
gbv_hate_speech_output = keras.layers.Dense(3, activation = 'softmax', name = 'gbv_hate_speech_output')(gbv_hate_speech_features)

In [68]:
# Compile the model with multiple inputs and outputs
model = keras.models.Model(inputs = [emotion_input_layer, violence_input_layer, gbv_hate_speech_input_layer],
                           outputs = [emotion_output, violence_output, gbv_hate_speech_output])

model.compile(optimizer = 'adam',
              loss = {
                  'emotion_output':'sparse_categorical_crossentropy',
                  'violence_output':'sparse_categorical_crossentropy',
                  'gbv_hate_speech_output':'sparse_categorical_crossentropy'
              },
              metrics = {
                  'emotion_output':'accuracy',
                  'violence_output': 'accuracy',
                  'gbv_hate_speech_output': 'accuracy'
              })

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ emotion_input       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ violence_input      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gbv_hate_speech_in… │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 50, 128)   │  5,357,312 │ emotion_input[0]… │
│ (Embedding)         │                   │            │ violence_input[0… │
│                     │                   │            │ gbv_hate_speech_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ (None, 50, 64)    │     49,408 │ embedding_12[0][… │
│                     │                   │            │ embedding_12[1][… │
│                     │                   │            │ embedding_12[2][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ lstm_11[0][0],    │
│ (GlobalAveragePool… │                   │            │ lstm_11[1][0],    │
│                     │                   │            │ lstm_11[2][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 64)        │          0 │ global_average_p… │
│ (Dropout)           │                   │            │ global_average_p… │
│                     │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emotion_output      │ (None, 6)         │        390 │ dropout_10[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ violence_output     │ (None, 5)         │        325 │ dropout_10[1][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gbv_hate_speech_ou… │ (None, 3)         │        195 │ dropout_10[2][0]  │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,407,630 (20.63 MB)

 Trainable params: 5,407,630 (20.63 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training the model with separate inputs.
model.fit(x = {
    'emotion_input': emotion_input,
    'violence_input' : violence_input,
    'gbv_hate_speech_input': gbv_hate_speech_input},
          y = {
    'emotion_output': emotion_labels,
    'violence_output' : violence_labels,
    'gbv_hate_speech_output': gbv_hate_speech_labels},
    epochs = 10, batch_size = 4)


Epoch 1/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 429s 141ms/step - emotion_output_accuracy: 0.2039 - emotion_output_loss: 1.7778 - gbv_hate_speech_output_accuracy: 0.6826 - gbv_hate_speech_output_loss: 0.7630 - loss: 2.9772 - violence_output_accuracy: 0.8470 - violence_output_loss: 0.4364
Epoch 2/10
 102/3000 ━━━━━━━━━━━━━━━━━━━━ 6:55 143ms/step - emotion_output_accuracy: 0.7038 - emotion_output_loss: 0.8446 - gbv_hate_speech_output_accuracy: 0.8277 - gbv_hate_speech_output_loss: 0.4677 - loss: 1.3532 - violence_output_accuracy: 0.9799 - violence_output_loss: 0.0410